In [1]:
import time
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("PySpark_ALS") \
    .master("local[*]") \
    .getOrCreate()
print(f"Spark Session успешно создана! Версия Spark: {spark.version}")

Spark Session успешно создана! Версия Spark: 3.5.1


# Разведочный анализ данных

Book-Crossing Dataset — история оценок книг от пользователей маркетплейса Book-Crossing. Этот набор данных состоит из трёх файлов:
- `Ratings.csv` — оценки пользователей. `Book-Rating = 0` — неявная оценка (факт взаимодействия),
  `1..10` — явная оценка (чем выше, тем лучше).
- `Books.csv` — метаданные книг: ISBN, автор, название, год, издательство.
- `Users.csv` — демография пользователей: локация и возраст (могут быть NULL).

In [3]:
ratings = spark.read.csv("data/Ratings.csv", header=True, inferSchema=True)
books = spark.read.csv("data/Books.csv", header=True, inferSchema=True)
users = spark.read.csv("data/Users.csv", header=True, inferSchema=True)

In [4]:
ratings.printSchema()
ratings.show(5, truncate=False)

root
 |-- User-ID: integer (nullable = true)
 |-- ISBN: string (nullable = true)
 |-- Book-Rating: integer (nullable = true)

+-------+----------+-----------+
|User-ID|ISBN      |Book-Rating|
+-------+----------+-----------+
|276725 |034545104X|0          |
|276726 |0155061224|5          |
|276727 |0446520802|0          |
|276729 |052165615X|3          |
|276729 |0521795028|6          |
+-------+----------+-----------+
only showing top 5 rows



In [5]:
books.printSchema()
books.show(5, truncate=False)

root
 |-- ISBN: string (nullable = true)
 |-- Book-Title: string (nullable = true)
 |-- Book-Author: string (nullable = true)
 |-- Year-Of-Publication: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- Image-URL-S: string (nullable = true)
 |-- Image-URL-M: string (nullable = true)
 |-- Image-URL-L: string (nullable = true)

+----------+--------------------------------------------------------------------------------------------------+--------------------+-------------------+--------------------------+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------------------------+
|ISBN      |Book-Title                                                                                        |Book-Author         |Year-Of-Publication|Publisher                 |Image-URL-S                                                 |Image-URL-M                                      

In [6]:
users.printSchema()
users.show(5, truncate=False)

root
 |-- User-ID: integer (nullable = true)
 |-- Location: string (nullable = true)
 |-- Age: string (nullable = true)

+-------+----------------------------------+----+
|User-ID|Location                          |Age |
+-------+----------------------------------+----+
|1      |nyc, new york, usa                |NULL|
|2      |stockton, california, usa         |18.0|
|3      |moscow, yukon territory, russia   |NULL|
|4      |porto, v.n.gaia, portugal         |17.0|
|5      |farnborough, hants, united kingdom|NULL|
+-------+----------------------------------+----+
only showing top 5 rows



In [7]:
print("Количество пропусков в Ratings.csv:")
ratings.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in ratings.columns]).show()
print("Количество пропусков в Books.csv:")
books.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in books.columns]).show()
print("Количество пропусков в Users.csv:")
users.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in users.columns]).show()

Количество пропусков в Ratings.csv:
+-------+----+-----------+
|User-ID|ISBN|Book-Rating|
+-------+----+-----------+
|      0|   0|          0|
+-------+----+-----------+

Количество пропусков в Books.csv:
+----+----------+-----------+-------------------+---------+-----------+-----------+-----------+
|ISBN|Book-Title|Book-Author|Year-Of-Publication|Publisher|Image-URL-S|Image-URL-M|Image-URL-L|
+----+----------+-----------+-------------------+---------+-----------+-----------+-----------+
|   0|         0|          1|                  0|        2|          0|          0|          3|
+----+----------+-----------+-------------------+---------+-----------+-----------+-----------+

Количество пропусков в Users.csv:
+-------+--------+------+
|User-ID|Location|   Age|
+-------+--------+------+
|      1|       0|110518|
+-------+--------+------+



In [8]:
isbns_in_ratings = ratings.select("ISBN").distinct()
isbns_in_books = books.select("ISBN").distinct()

n_isbns_ratings = isbns_in_ratings.count()
print(f"Уникальных ISBN в Ratings.csv: {n_isbns_ratings}")

n_missing_isbns = isbns_in_ratings.join(
    isbns_in_books, on="ISBN", how="left_anti"
).count()
print(
    f"ISBN из Ratings.csv, отсутствующих в Books.csv: {n_missing_isbns} "
    f"({n_missing_isbns / n_isbns_ratings * 100:.2f}%)"
)

print(
    "ISBN из Books.csv, отсутствующих в Ratings.csv (книги без оценок): "
    f"{isbns_in_books.count() - (n_isbns_ratings - n_missing_isbns)}"
)

Уникальных ISBN в Ratings.csv: 340556
ISBN из Ratings.csv, отсутствующих в Books.csv: 70405 (20.67%)
ISBN из Books.csv, отсутствующих в Ratings.csv (книги без оценок): 1209


In [9]:
n_ratings = ratings.count()
n_users = ratings.select("User-ID").distinct().count()
n_books = ratings.select("ISBN").distinct().count()
n_explicit = ratings.filter(F.col("Book-Rating") > 0).count()
n_implicit = n_ratings - n_explicit

print(
    f"Всего оценок: {n_ratings}",
    f"- явных (1..10): {n_explicit} ({n_explicit / n_ratings * 100:.2f}%)",
    f"- неявных (0): {n_implicit} ({n_implicit / n_ratings * 100:.2f}%)",
    f"\nУникальных пользователей (с оценками): {n_users}",
    f"Уникальных книг (с оценками): {n_books}",
    f"\nСреднее количество оценок на пользователя: {n_ratings / n_users:.1f}",
    f"Среднее количество оценок на книгу: {n_ratings / n_books:.1f}",
    f"\nПлотность матрицы: {n_ratings / (n_users * n_books) * 100:.4f}%",
    sep="\n",
)

Всего оценок: 1149780
- явных (1..10): 433671 (37.72%)
- неявных (0): 716109 (62.28%)

Уникальных пользователей (с оценками): 105283
Уникальных книг (с оценками): 340556

Среднее количество оценок на пользователя: 10.9
Среднее количество оценок на книгу: 3.4

Плотность матрицы: 0.0032%


In [10]:
n_dup_ratings = ratings.groupBy("User-ID", "ISBN").count().filter(F.col("count") > 1).count()
print(f"Дубликаты отзывов по паре User-ID и ISBN: {n_dup_ratings}")

n_dup_books = books.groupBy("ISBN").count().filter(F.col("count") > 1).count()
print(f"Дубликаты книг по ISBN: {n_dup_books}")

Дубликаты отзывов по паре User-ID и ISBN: 0
Дубликаты книг по ISBN: 0


# Реализация базовой эвристики «Популярное»

In [11]:
book_stats = ratings.groupBy("ISBN").agg(
    F.count("*").alias("total_ratings"),
    F.count(F.when(F.col("Book-Rating") > 0, True)).alias("explicit_ratings"),
    F.round(F.avg(F.when(F.col("Book-Rating") > 0, F.col("Book-Rating"))), 2).alias("avg_rating"),
)

top_books = book_stats \
    .filter((F.col("explicit_ratings") >= 50) & (F.col("avg_rating").isNotNull())) \
    .orderBy(F.desc("avg_rating"), F.desc("total_ratings")) \
    .limit(10)
top_books_enriched = top_books.join(
    books.select("ISBN", "Book-Title", "Book-Author"), on="ISBN", how="left"
)

print("Топ-10 самых популярных книг (критерий: >= 50 оценок, сортировка по средней):")
top_books_enriched.select(
    "Book-Title", "Book-Author", "avg_rating", "explicit_ratings", "total_ratings",
).orderBy(F.desc("avg_rating")).show(truncate=False)

Топ-10 самых популярных книг (критерий: >= 50 оценок, сортировка по средней):
+------------------------------------------------------+-------------------------+----------+----------------+-------------+
|Book-Title                                            |Book-Author              |avg_rating|explicit_ratings|total_ratings|
+------------------------------------------------------+-------------------------+----------+----------------+-------------+
|The Return of the King (The Lord of the Rings, Part 3)|J.R.R. TOLKIEN           |9.4       |77              |173          |
|Harry Potter and the Goblet of Fire (Book 4)          |J. K. Rowling            |9.26      |137             |194          |
|Harry Potter and the Sorcerer's Stone (Book 1)        |J. K. Rowling            |9.21      |53              |104          |
|The Two Towers (The Lord of the Rings, Part 2)        |J.R.R. TOLKIEN           |9.12      |83              |177          |
|Harry Potter and the Prisoner of Azkaban (Book

**Недостатки такого подхода:**

- *Нет персонализации.* Все пользователи получают одинаковый список. Для крупного маркетплейса это означает, что мы фактически не используем ни одну из имеющихся у нас сигнатур предпочтений.

- *Popularity bias.* Книги, которые уже набрали много оценок, остаются наверху; книги с историей короче никогда не попадают в топ. Это искажает статистику и усиливает монополию бестселлеров.

- *Не решает задачу длинного хвоста.* Основной оборот и разнообразие каталога — именно в нишевых книгах, а они в этой эвристике отсутствуют по определению.

- *Холодный старт каталога.* У новой книги нет оценок, значит она не может попасть в топ в принципе, как бы хороша ни была.

- *Не учитывает контекст.* Топ не меняется ни от сессии, ни от времени суток, ни от страны пользователя.

# Коллаборативная фильтрация вручную

In [12]:
ratings_explicit = ratings.filter(F.col("Book-Rating") > 0)
user_activity = ratings_explicit.groupBy("User-ID").count().orderBy(F.desc("count"))
top_users = user_activity.limit(10)

top_users_with_info = top_users \
    .join(users, on="User-ID", how="left") \
    .withColumnRenamed("count", "Ratings") \
    .select("User-ID", "Location", "Age", "Ratings") \
    .orderBy(F.desc("Ratings"))

print("Топ-10 самых активных пользователей:")
top_users_with_info.show(10, truncate=False)

Топ-10 самых активных пользователей:
+-------+-------------------------------+----+-------+
|User-ID|Location                       |Age |Ratings|
+-------+-------------------------------+----+-------+
|11676  |n/a, n/a, n/a                  |NULL|8524   |
|98391  |morrow, georgia, usa           |52.0|5802   |
|153662 |ft. stewart, georgia, usa      |44.0|1969   |
|189835 |honolulu, hawaii, usa          |NULL|1906   |
|23902  |london, england, united kingdom|NULL|1395   |
|76499  |new york, new york, usa        |NULL|1036   |
|171118 |toronto, ontario, canada       |47.0|1035   |
|235105 |st louis, missouri, usa        |46.0|1023   |
|16795  |mechanicsville, maryland, usa  |47.0|968    |
|248718 |hyde park, new york, usa       |43.0|948    |
+-------+-------------------------------+----+-------+



In [13]:
quantiles_user_activity = [0.0, 0.5, 0.75, 0.85, 0.95, 0.97, 0.98, 0.985, 0.9875, 0.99, 1.0]
df_quantiles_user_activity = spark.createDataFrame(
    [(float(q), float(v)) for q, v in zip(
        quantiles_user_activity,
        user_activity.approxQuantile("count", quantiles_user_activity, 0.01)
    )],
    ["Quantile", "Ratings"],
)
print("Квантили числа явных оценок на пользователя:")
df_quantiles_user_activity.orderBy("Quantile").show()

avg_ratings = user_activity.agg(F.round(F.avg("count"), 1)).first()[0]
print(f"Среднее число явных оценок на пользователя: {avg_ratings}")

Квантили числа явных оценок на пользователя:
+--------+-------+
|Quantile|Ratings|
+--------+-------+
|     0.0|    1.0|
|     0.5|    1.0|
|    0.75|    3.0|
|    0.85|    6.0|
|    0.95|   17.0|
|    0.97|   27.0|
|    0.98|   34.0|
|   0.985|   45.0|
|  0.9875|   45.0|
|    0.99| 8524.0|
|     1.0| 8524.0|
+--------+-------+

Среднее число явных оценок на пользователя: 5.6


In [14]:
target_user_id = 248718
print(f"Целевой пользователь: {target_user_id}")

target_ratings = ratings_explicit.filter(F.col("User-ID") == target_user_id)
n_target = target_ratings.count()
target_avg = target_ratings.agg(F.round(F.avg("Book-Rating"), 2)).first()[0]
print(f"Явных оценок: {n_target}", f"Средняя оценка: {target_avg}", sep="\n")

target_ratings.show(10, truncate=False)

Целевой пользователь: 248718
Явных оценок: 948
Средняя оценка: 7.95
+-------+----------+-----------+
|User-ID|ISBN      |Book-Rating|
+-------+----------+-----------+
|248718 |000433549X|9          |
|248718 |0020103514|8          |
|248718 |0020225601|5          |
|248718 |002026478X|10         |
|248718 |0020383142|9          |
|248718 |0025876007|9          |
|248718 |0028604105|7          |
|248718 |0028604954|8          |
|248718 |0028617274|10         |
|248718 |0060157917|8          |
+-------+----------+-----------+
only showing top 10 rows



In [15]:
target_pairs = target_ratings.select(
    F.col("ISBN"),
    F.col("Book-Rating").alias("target_rating"),
)

ratings_overlap = ratings_explicit \
    .filter(F.col("User-ID") != target_user_id) \
    .select(
        F.col("User-ID").alias("neighbor_id"),
        F.col("ISBN"),
        F.col("Book-Rating").alias("neighbor_rating"),
    ) \
    .join(target_pairs, on="ISBN", how="inner")

target_neighbors = ratings_overlap.groupBy("neighbor_id").agg(
    F.count("*").alias("common_books"),
    F.round(F.avg(F.abs(
        F.col("neighbor_rating") - F.col("target_rating")
    )), 2).alias("ratings_diff"),
).orderBy(F.desc("common_books")).limit(3)

neighbor_activity = user_activity.withColumnRenamed("User-ID", "neighbor_id") \
    .withColumnRenamed("count", "total_ratings")

neighbors_enriched = target_neighbors \
    .join(neighbor_activity, on="neighbor_id", how="left") \
    .select("neighbor_id", "total_ratings", "common_books", "ratings_diff") \
    .orderBy(F.desc("common_books"))

print(f"Топ-3 соседа для пользователя {target_user_id}:")
neighbors_enriched.show()

Топ-3 соседа для пользователя 248718:
+-----------+-------------+------------+------------+
|neighbor_id|total_ratings|common_books|ratings_diff|
+-----------+-------------+------------+------------+
|      11676|         8524|          45|         1.8|
|     257204|          476|          27|        1.74|
|     153662|         1969|          19|        2.05|
+-----------+-------------+------------+------------+



In [16]:
neighbor_ids = [row["neighbor_id"] for row in target_neighbors.collect()]

neighbors_high = ratings_explicit \
    .filter(F.col("User-ID").isin(neighbor_ids)) \
    .filter(F.col("Book-Rating") >= 8) \
    .select(
        F.col("User-ID").alias("neighbor_id"),
        F.col("ISBN"),
        F.col("Book-Rating").alias("neighbor_rating"),
    )

books_candidates = neighbors_high \
    .join(target_ratings.select("ISBN").distinct(), on="ISBN", how="left_anti") \
    .groupBy("ISBN") \
    .agg(
        F.count("*").alias("neighbors_count"),
        F.round(F.avg("neighbor_rating"), 2).alias("avg_neighbor_rating"),
    ) \
    .orderBy(F.desc("neighbors_count"), F.desc("avg_neighbor_rating")) \
    .limit(10)

target_recommendations = books_candidates.join(
    books.select("ISBN", "Book-Title", "Book-Author"), on="ISBN", how="left"
)

print(f"Топ-10 рекомендаций для пользователя {target_user_id}:")
target_recommendations.select(
    "Book-Title", "Book-Author", "neighbors_count", "avg_neighbor_rating",
).show(10, truncate=False)

Топ-10 рекомендаций для пользователя 248718:
+----------------------------------------------------------------------------+------------------------+---------------+-------------------+
|Book-Title                                                                  |Book-Author             |neighbors_count|avg_neighbor_rating|
+----------------------------------------------------------------------------+------------------------+---------------+-------------------+
|Key of Valor (Roberts, Nora. Key Trilogy, 3.)                               |Nora Roberts            |2              |10.0               |
|Dragons of Spring Dawning (Dragonlance Chronicles, Vol 3)                   |Margaret Weis           |2              |10.0               |
|Moreta: Dragonlady of Pern                                                  |Anne McCaffrey          |2              |10.0               |
|My Sergei : A Love Story                                                    |E. M. Swift             |2           

**Почему в PySpark нет встроенного поиска соседей:**

- *Сложность `O(N²)`* по числу пользователей: для 278k пользователей это ~77 млрд парных сравнений.

- *Self-join плохо параллелится:* каждый пользователь должен «встретиться» с каждым, данные перебрасываются между узлами на каждой итерации.

- *Плотная матрица сходства `users × users` не влезает в память* ни в один узел, ни на диск (в нашем случае — сотни гигабайт).

- *Нет инкрементальных обновлений:* при появлении новых пользователей и книг матрицу пришлось бы пересчитывать целиком.

- *Разреженность:* у большинства пользователей 1–5 оценок, 99% парных сравнений дают пустое пересечение, но стоят столько же, сколько полезные.

Алгоритм **ALS** решает эти проблемы через низкоразмерные факторные представления: сходство становится скалярным произведением коротких векторов — `O(N·K)` вместо `O(N²)`.

**Недостатки наивного поиска соседей:**

- *Критерий сходства примитивный.* Мы считаем просто число общих книг и не учитываем ни разброс оценок, ни их распределение.

- *Соседи — не эксперты.* Активный пользователь может оценивать всё подряд, включая случайные книги. Его «высокие» оценки не обязательно означают хорошие рекомендации.

- *Нет нормализации на популярность.* Книга, которую читали все, автоматически получит больше «голосов» соседей, даже если она посредственная.

- *Нет отказоустойчивости к холодному пользователю.* Для нового пользователя пересечений ни с кем нет, и весь пайплайн обнуляется.

# Коллаборативная фильтрация с помощью ALS

In [17]:
ratings_als = ratings_explicit \
    .select("User-ID", "ISBN", "Book-Rating") \
    .withColumnRenamed("User-ID", "userId") \
    .withColumnRenamed("Book-Rating", "rating")
print(f"Количество оценок для ALS: {ratings_als.count()}")

isbn_indexer = StringIndexer(inputCol="ISBN", outputCol="bookId", handleInvalid="keep")
isbn_indexer_model = isbn_indexer.fit(ratings_als)
ratings_als = isbn_indexer_model.transform(ratings_als)
print("\nСхема данных, подготовленных для ALS:")
ratings_als.printSchema()

train_data, test_data = ratings_als.randomSplit([0.8, 0.2], seed=0)
train_data.cache()
test_data.cache()
print(f"Обучающая выборка: {train_data.count()} строк")
print(f"Тестовая выборка:  {test_data.count()} строк")

Количество оценок для ALS: 433671

Схема данных, подготовленных для ALS:
root
 |-- userId: integer (nullable = true)
 |-- ISBN: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- bookId: double (nullable = false)

Обучающая выборка: 346906 строк
Тестовая выборка:  86765 строк


In [18]:
als = ALS(
    userCol="userId", itemCol="bookId", ratingCol="rating",
    rank=10, maxIter=10, regParam=0.1, nonnegative=True,
    coldStartStrategy="drop", seed=0,
)

print(f"Обучение модели ALS:")
t0 = time.time()
model_als = als.fit(train_data)
t_fit = time.time() - t0
print(f"- время обучения: {round(t_fit, 3)} сек.")

t1 = time.time()
preds = model_als.transform(test_data)
preds.cache()
_ = preds.count()
t_pred = time.time() - t1
print(f"- время предсказания: {round(t_pred, 3)} сек.")

Обучение модели ALS:
- время обучения: 16.082 сек.
- время предсказания: 12.229 сек.


In [19]:
preds.select("userId", "ISBN", "rating", "prediction").show(10, truncate=False)

+------+----------+------+----------+
|userId|ISBN      |rating|prediction|
+------+----------+------+----------+
|48494 |080410753X|10    |8.717929  |
|44611 |080410753X|10    |9.350386  |
|11933 |080410753X|7     |2.8005328 |
|49635 |080410753X|7     |7.852251  |
|1083  |080410753X|7     |9.921227  |
|69971 |080410753X|10    |7.0698996 |
|59209 |080410753X|8     |9.152082  |
|70052 |080410753X|9     |6.680977  |
|135149|080410753X|7     |7.714522  |
|111749|080410753X|7     |7.6667333 |
+------+----------+------+----------+
only showing top 10 rows



In [20]:
print("Результаты на тестовой выборке:")

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction",
)
rmse_als = evaluator.evaluate(preds)
print(f"RMSE модели ALS: {rmse_als:.4f}")

global_mean = train_data.agg(F.round(F.avg("rating"), 4)).first()[0]
print(f"Глобальное среднее по рейтингу: {global_mean}")

baseline_preds = test_data.withColumn("prediction", F.lit(global_mean))
rmse_baseline = evaluator.evaluate(baseline_preds)
print(f"RMSE глобального среднего: {rmse_baseline:.4f}")

print(
    "Улучшение ALS относительно глобального среднего: "
    f"{(rmse_baseline - rmse_als) / rmse_baseline * 100:.2f}%"
)

Результаты на тестовой выборке:
RMSE модели ALS: 2.5021
Глобальное среднее по рейтингу: 7.602
RMSE глобального среднего: 1.8468
Улучшение ALS относительно глобального среднего: -35.49%


- *Холодный пользователь.* У ALS нет для него вектора скрытых факторов, поэтому модель не может предсказать ни одной оценки — `prediction = NaN`. При `coldStartStrategy="drop"` такие строки автоматически отфильтровываются.

- *Пользователь с одной оценкой.* Если он встречается в обучающей выборке хотя бы один раз, ALS обучит для него вектор. То есть **одной оценки достаточно** для получения предсказаний, но качество этих предсказаний будет близко к «среднему по больнице», потому что одна оценка почти ничего не говорит о вкусе.

In [21]:
cold_users = ratings_als.select("userId").distinct() \
    .join(train_data.select("userId").distinct(), on="userId", how="left_anti")
cold_users_with_counts = cold_users \
    .join(test_data.groupBy("userId").count(), on="userId", how="inner") \
    .filter(F.col("count") >= 5)
cold_user_id = cold_users_with_counts.first()["userId"]
print(f"Пример холодного пользователя: {cold_user_id}")

cold_input = test_data.filter(F.col("userId") == cold_user_id) \
    .select("userId", "bookId", "rating").limit(5)

print("\nПредсказание с coldStartStrategy='drop':")
model_als.transform(cold_input).show()

als_no_drop = ALS(
    userCol="userId", itemCol="bookId", ratingCol="rating",
    rank=10, maxIter=10, regParam=0.1, nonnegative=True,
    coldStartStrategy="nan", seed=0,
)

print(f"Обучение модели ALS с coldStartStrategy='nan':")
t0 = time.time()
model_als_no_drop = als_no_drop.fit(train_data)
t_fit = time.time() - t0
print(f"- время обучения: {round(t_fit, 3)} сек.")

print("\nПредсказание с coldStartStrategy='nan':")
model_als_no_drop.transform(cold_input).show(truncate=False)

Пример холодного пользователя: 90170

Предсказание с coldStartStrategy='drop':
+------+------+------+----------+
|userId|bookId|rating|prediction|
+------+------+------+----------+
+------+------+------+----------+

Обучение модели ALS с coldStartStrategy='nan':
- время обучения: 14.509 сек.

Предсказание с coldStartStrategy='nan':
+------+--------+------+----------+
|userId|bookId  |rating|prediction|
+------+--------+------+----------+
|90170 |4000.0  |6     |NaN       |
|90170 |126073.0|8     |NaN       |
|90170 |101922.0|10    |NaN       |
|90170 |113135.0|7     |NaN       |
|90170 |3315.0  |9     |NaN       |
+------+--------+------+----------+



In [22]:
single_users = train_data.groupBy("userId").count().filter(F.col("count") == 1)
single_users_in_test = single_users \
    .join(test_data.select("userId").distinct(), on="userId", how="inner") \
    .limit(5)
print(
    f"Примеры пользователей с одной оценкой: "
    f"{[r['userId'] for r in single_users_in_test.collect()]}"
)

single_user_ids = [r["userId"] for r in single_users_in_test.collect()]
single_test_ratings = test_data \
    .filter(F.col("userId").isin(single_user_ids)) \
    .select("userId", "bookId", "rating")

print("\nПредсказания модели для таких пользователей:")
model_als.transform(single_test_ratings).show(truncate=False)   

Примеры пользователей с одной оценкой: [4900, 31983, 36525, 45615, 47711]

Предсказания модели для таких пользователей:
+------+-------+------+----------+
|userId|bookId |rating|prediction|
+------+-------+------+----------+
|36525 |9514.0 |8     |5.1016183 |
|45615 |4333.0 |6     |4.6927624 |
|31983 |42732.0|5     |0.15613003|
|4900  |51.0   |7     |5.504751  |
|47711 |20587.0|8     |3.6471167 |
+------+-------+------+----------+



In [23]:
preds_enriched = preds \
    .select("userId", "ISBN", "bookId", "rating", "prediction") \
    .join(books.select("ISBN", "Book-Title", "Book-Author"), on="ISBN", how="left")
print("Примеры предсказаний ALS с названиями и авторами книг:")
preds_enriched.select("userId", "Book-Title", "Book-Author", "rating", "prediction") \
    .show(10, truncate=False)

Примеры предсказаний ALS с названиями и авторами книг:
+------+------------------------------------------------------------------------+-------------+------+----------+
|userId|Book-Title                                                              |Book-Author  |rating|prediction|
+------+------------------------------------------------------------------------+-------------+------+----------+
|20032 |Stupid White Men ...and Other Sorry Excuses for the State of the Nation!|Michael Moore|2     |6.9488606 |
|4157  |Stupid White Men ...and Other Sorry Excuses for the State of the Nation!|Michael Moore|8     |7.8419466 |
|36570 |Stupid White Men ...and Other Sorry Excuses for the State of the Nation!|Michael Moore|3     |6.6454515 |
|16634 |Stupid White Men ...and Other Sorry Excuses for the State of the Nation!|Michael Moore|8     |6.689911  |
|5048  |Stupid White Men ...and Other Sorry Excuses for the State of the Nation!|Michael Moore|10    |8.164649  |
|36907 |Stupid White Men ...and O

In [24]:
top_pred_books = preds_enriched.groupBy("ISBN", "Book-Title", "Book-Author") \
    .agg(
        F.count("*").alias("n_preds"),
        F.round(F.avg("prediction"), 2).alias("avg_pred_rating"),
    ) \
    .filter(F.col("n_preds") >= 7) \
    .orderBy(F.desc("avg_pred_rating"))
print("Топ-10 книг по средней прогнозной оценке:")
top_pred_books.show(10, truncate=False)

Топ-10 книг по средней прогнозной оценке:
+----------+----------------------------------------------------------+-------------------------+-------+---------------+
|ISBN      |Book-Title                                                |Book-Author              |n_preds|avg_pred_rating|
+----------+----------------------------------------------------------+-------------------------+-------+---------------+
|0345339711|The Two Towers (The Lord of the Rings, Part 2)            |J.R.R. TOLKIEN           |9      |9.55           |
|0441627404|The Once and Future King                                  |T. H. White              |7      |9.35           |
|0345339738|The Return of the King (The Lord of the Rings, Part 3)    |J.R.R. TOLKIEN           |13     |9.33           |
|0156012197|The Little Prince                                         |Antoine de Saint-ExupÃ©ry|8      |9.29           |
|0618002227|The Fellowship of the Ring (The Lord of the Rings, Part 1)|J. R. R. Tolkien         |8      

**Метрика RMSE** измеряет среднюю ошибку в баллах при предсказании рейтинга.

Для задачи рекомендаций это **не идеальная метрика**, потому что:

- *RMSE — про регрессию, а не про ранжирование.* Нам не важно, предсказали мы 7.2 или 7.5 — важно, чтобы правильные книги оказались наверху списка. RMSE этого не различает.
- *Ошибки на «холодных» и «горячих» объектах не различаются.* Одна крупная ошибка на популярной книге и такая же на редкой дают одинаковый вклад в метрику, хотя ценность этих ошибок разная.
- *RMSE не отличает «случайно угадал» от «действительно знает вкус».* Если пользователь всегда ставит 8, модель, выдающая везде 8, получит хороший RMSE, но не будет ничего рекомендовать.
- *RMSE не коррелирует с бизнес-метриками.* В индустрии качество рекомендаций, чаще всего, измеряют через *Precision(K), Recall(K), Coverage, Novelty* — они оценивают именно то, что важно продукту: попадёт ли релевантная книга в топ-K и насколько разнообразны рекомендации.

RMSE, тем не менее, полезна как **быстрая диагностика**: она показывает, что ALS в принципе выучил что-то сверх среднего. Сравнение с глобальным средним помогает это проверить.

# Гибридный подход к рекомендациям

In [25]:
hot_user_id = target_user_id

hot_user_df = spark.createDataFrame([(hot_user_id,)], ["userId"])
hot_user_recs = model_als.recommendForUserSubset(hot_user_df, 3)

hot_user_recs_flat = hot_user_recs.select(
    "userId",
    F.explode("recommendations").alias("rec"),
).select(
    F.col("rec.bookId").alias("bookId"),
    F.round(
        F.least(F.greatest(F.col("rec.rating"), F.lit(1.0)), F.lit(10.0)), 2
    ).alias("pred_rating"),
)

hot_user_recs_named = hot_user_recs_flat \
    .join(ratings_als.select("bookId", "ISBN").distinct(), on="bookId", how="left") \
    .join(books.select("ISBN", "Book-Title", "Book-Author"), on="ISBN", how="left") \
    .select("ISBN", "Book-Title", "Book-Author", "pred_rating") \
    .orderBy(F.desc("pred_rating"))

print(f"Топ-3 рекомендации для «старого» пользователя (через ALS):")
hot_user_recs_named.show(truncate=False)

Топ-3 рекомендации для «старого» пользователя (через ALS):
+----------+----------+--------------+-----------+
|ISBN      |Book-Title|Book-Author   |pred_rating|
+----------+----------+--------------+-----------+
|0380708388|Warbirds  |Richard Herman|10.0       |
|472612453 |NULL      |NULL          |10.0       |
|033039892X|The Stolen|Alex Shearer  |10.0       |
+----------+----------+--------------+-----------+



In [26]:
print("Топ-3 рекомендации для «холодного» пользователя (через популярное):")
top_books_enriched.select(
    "Book-Title", "Book-Author", "avg_rating", "explicit_ratings", "total_ratings"
).orderBy(F.desc("avg_rating")).limit(3).show(truncate=False)

Топ-3 рекомендации для «холодного» пользователя (через популярное):
+------------------------------------------------------+--------------+----------+----------------+-------------+
|Book-Title                                            |Book-Author   |avg_rating|explicit_ratings|total_ratings|
+------------------------------------------------------+--------------+----------+----------------+-------------+
|The Return of the King (The Lord of the Rings, Part 3)|J.R.R. TOLKIEN|9.4       |77              |173          |
|Harry Potter and the Goblet of Fire (Book 4)          |J. K. Rowling |9.26      |137             |194          |
|Harry Potter and the Sorcerer's Stone (Book 1)        |J. K. Rowling |9.21      |53              |104          |
+------------------------------------------------------+--------------+----------+----------------+-------------+



**Как улучшить рекомендации для «холодного» пользователя.**

Эвристика «Популярное» — самое простое решение. Если у нас есть хоть какая-то информация
о новом пользователе, её стоит использовать:

- *Контентная фильтрация по одной оценке.* Если пользователь поставил высокую оценку хотя бы одной книге, можно взять её `Book-Author` или `Publisher` и рекомендовать другие высокорейтинговые книги тех же авторов или издательств, исключая уже оценённые.

- *Демографическая сегментация.* По `Age` или `Location` можно выделять группы «похожих» пользователей и находить популярные книги внутри них. Новому пользователю с известным возрастом можно показывать топ книг его возврастной группы — это уже не «популярное вообще», а «популярное среди похожих на него».

- *Контекст текущей сессии.* Если пользователь открыл карточку книги — показываем книги того же автора/жанра. Это уже не рекомендация, а «похожие товары», но как первый экран для новичка работает лучше, чем абстрактный топ.

In [27]:
sample_isbn = "0618002227"
sample_book = books.filter(F.col("ISBN") == sample_isbn) \
    .select("ISBN", "Book-Title", "Book-Author").first()
print(
    "Предположим, что пользователь высоко оценил книгу:",
    f"{sample_book['Book-Title']} под авторством {sample_book['Book-Author']}",
    sep="\n",
)

book_quality = ratings_explicit.groupBy("ISBN").agg(
    F.count("*").alias("n_ratings"),
    F.round(F.avg("Book-Rating"), 2).alias("avg_rating"),
)

recs_by_author = books \
    .filter(F.col("Book-Author") == sample_book["Book-Author"]) \
    .filter(F.col("ISBN") != sample_isbn) \
    .select("ISBN", "Book-Title", "Book-Author", "Year-of-Publication") \
    .join(book_quality, on="ISBN", how="inner") \
    .filter(F.col("n_ratings") >= 10) \
    .orderBy(F.desc("avg_rating")) \
    .limit(5)

print("\nКонтентные рекомендации новому пользователю (по автору):")
recs_by_author.show(truncate=False)

Предположим, что пользователь высоко оценил книгу:
The Fellowship of the Ring (The Lord of the Rings, Part 1) под авторством J. R. R. Tolkien

Контентные рекомендации новому пользователю (по автору):
+----------+------------------------------------------------------+----------------+-------------------+---------+----------+
|ISBN      |Book-Title                                            |Book-Author     |Year-of-Publication|n_ratings|avg_rating|
+----------+------------------------------------------------------+----------------+-------------------+---------+----------+
|0618002235|The Two Towers (The Lord of the Rings, Part 2)        |J. R. R. Tolkien|1999               |25       |9.72      |
|0618002243|The Return of the King (The Lord of The Rings, Part 3)|J. R. R. Tolkien|1999               |16       |9.63      |
|0618129081|The Two Towers (The Lord of the Rings, Part 2)        |J. R. R. Tolkien|2001               |12       |9.33      |
|0345272579|The Hobbit                      

In [28]:
target_user_age = 50
print(
    "Предположим, что целевой пользователь указал "
    f"свой возраст: {target_user_age} лет"
)

age_diff = 5
age_min, age_max = target_user_age - age_diff, target_user_age + age_diff

users_age_group = users \
    .filter(F.col("Age").isNotNull()) \
    .filter((F.col("Age") >= age_min) & (F.col("Age") <= age_max)) \
    .select("User-ID")
print(
    "Пользователей с известным возрастом в группе "
    f"{age_min}-{age_max}: {users_age_group.count()}"
)

age_group_stats = ratings_explicit \
    .join(users_age_group, on="User-ID", how="inner") \
    .groupBy("ISBN") \
    .agg(
        F.count("*").alias("n_ratings"),
        F.round(F.avg("Book-Rating"), 2).alias("avg_rating"),
    ) \
    .filter(F.col("n_ratings") >= 30) \
    .orderBy(F.desc("avg_rating")) \
    .limit(5) \
    .join(books.select("ISBN", "Book-Title", "Book-Author"), on="ISBN", how="left") \
    .select("ISBN", "Book-Title", "Book-Author", "n_ratings", "avg_rating") \
    .orderBy(F.desc("avg_rating"))

print(f"\nТоп-5 книг для возрастной группы {age_min}-{age_max}:")
age_group_stats.show(truncate=False)

Предположим, что целевой пользователь указал свой возраст: 50 лет
Пользователей с известным возрастом в группе 45-55: 24967

Топ-5 книг для возрастной группы 45-55:
+----------+------------------------------------------------------------+----------------------+---------+----------+
|ISBN      |Book-Title                                                  |Book-Author           |n_ratings|avg_rating|
+----------+------------------------------------------------------------+----------------------+---------+----------+
|0142001740|The Secret Life of Bees                                     |Sue Monk Kidd         |45       |8.64      |
|0385504209|The Da Vinci Code                                           |Dan Brown             |77       |8.44      |
|0679781587|NULL                                                        |NULL                  |33       |8.39      |
|0316666343|The Lovely Bones: A Novel                                   |Alice Sebold          |76       |8.11      |
|14000347

**Что делать с «холодными» книгами.**

ALS не рекомендует книги, которых не было в обучающей выборке, потому что у них нет латентного вектора. Это прямая потеря для бизнеса: новый контент не доходит до аудитории. Возможные стратегии:

- *Контентное проталкивание.* Находим пользователей, у которых в истории много книг того же автора или издательства, и принудительно добавляем новинку в их выдачу. Это уже не ALS-рекомендация, а надстройка над ним по метаданным.

- *Exploration budget.* Выделяем фиксированную долю показов (например, 5–10%) под новые книги — случайным или «похожим» пользователям. Смысл в том, чтобы накопить первые оценки. Как только книга наберёт минимальную критическую массу взаимодействий, при следующем переобучении ALS она получит вектор и войдёт в персональные рекомендации.

- *Явное продвижение на главной.* Витрины «Новинки», «Рекомендуем обратить внимание» работают в обход ALS и дают книге шанс набрать статистику независимо от модели.

Общий принцип: рекомендательная система должна не только эксплуатировать уже накопленные данные, но и активно собирать новые — иначе доля «холодных» книг будет только расти.

In [29]:
top_authors = books.groupBy("Book-Author").count().orderBy(F.desc("count"))
print("Топ авторов по количеству книг в каталоге:")
top_authors.show(10)

Топ авторов по количеству книг в каталоге:
+-------------------+-----+
|        Book-Author|count|
+-------------------+-----+
|    Agatha Christie|  631|
|William Shakespeare|  567|
|       Stephen King|  524|
|      Ann M. Martin|  423|
|      Carolyn Keene|  373|
|    Francine Pascal|  372|
|       Isaac Asimov|  330|
|       Nora Roberts|  315|
|   Barbara Cartland|  307|
|    Charles Dickens|  302|
+-------------------+-----+
only showing top 10 rows



In [30]:
top_author = books.groupBy("Book-Author").count() \
    .orderBy(F.desc("count")).first()["Book-Author"]
print(f"Автор с наибольшим числом книг в каталоге: {top_author}")

top_author_books = books.filter(F.col("Book-Author") == top_author)
top_author_coolest_books = top_author_books \
    .select("ISBN", "Book-Title", "Year-Of-Publication") \
    .join(
        ratings.groupBy("ISBN").count().withColumnRenamed("count", "n_ratings"),
        on="ISBN", how="left",
    ) \
    .fillna({"n_ratings": 0}) \
    .orderBy(F.asc("n_ratings"), F.asc("ISBN")) \
    .limit(10)
print("\nКниги этого автора с наименьшим числом оценок:")
top_author_coolest_books.show(truncate=False)

Автор с наибольшим числом книг в каталоге: Agatha Christie

Книги этого автора с наименьшим числом оценок:
+----------+----------------------------------------+-------------------+---------+
|ISBN      |Book-Title                              |Year-Of-Publication|n_ratings|
+----------+----------------------------------------+-------------------+---------+
|0671831771|EASY TO KILL                            |1979               |0        |
|0002315394|Murder on the Orient express            |1974               |1        |
|0002317850|Sleeping murder: Miss Marple's last case|1976               |1        |
|0061002755|The Seven Dials Mystery                 |1997               |1        |
|0061002771|Crooked House                           |1997               |1        |
|0061003700|Murder Is Easy                          |1992               |1        |
|0312330871|And Then There Were None                |2004               |1        |
|0312981619|Death Comes As the End                  |

In [32]:
fan_threshold = 2
top_author_fans = ratings_explicit \
    .join(top_author_books.select("ISBN"), on="ISBN", how="inner") \
    .groupBy("User-ID") \
    .agg(F.count("*").alias("n_top_author_rated_books")) \
    .filter(F.col("n_top_author_rated_books") >= fan_threshold) \
    .orderBy(F.desc("n_top_author_rated_books"))
print(
    f"Число пользователей, оценивших {fan_threshold} и более "
    f"книг автора {top_author}: {top_author_fans.count()}"
)
print(f"\nТоп самых активных читателей автора {top_author}:")
top_author_fans.show(10, truncate=False)

Число пользователей, оценивших 2 и более книг автора Agatha Christie: 109

Топ самых активных читателей автора Agatha Christie:
+-------+------------------------+
|User-ID|n_top_author_rated_books|
+-------+------------------------+
|125774 |38                      |
|254899 |31                      |
|11676  |29                      |
|23902  |27                      |
|10819  |23                      |
|109955 |17                      |
|43246  |15                      |
|203820 |12                      |
|95991  |11                      |
|264862 |9                       |
+-------+------------------------+
only showing top 10 rows

